# Stock Market Feature Engineering
**Linh Pham - Feature Engineering Module**

**Project**: Stock Market Return Prediction

## Tasks:
1. Create lagged features (past 1-, 5-, 10-day returns)
2. Compute technical indicators (MA, RSI, MACD, Bollinger Bands)
3. Add volatility and volume-based metrics
4. Standardize/normalize features
5. Create final feature matrix (X) and target vector (y) for each stock

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
from sklearn.preprocessing import StandardScaler, RobustScalerz

warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

## Configuration

In [ ]:
# Configuration
import os

STOCK_LIST = ['AAPL', 'GOOGL', 'MSFT', 'AMZN', 'TSLA']

# Find project root automatically
def find_project_root():
    search_dir = os.getcwd()
    for _ in range(5):
        if (os.path.exists(os.path.join(search_dir, 'data', 'processed', 'combined_stock_data.csv')) or
            os.path.exists(os.path.join(search_dir, 'experiment')) or
            os.path.exists(os.path.join(search_dir, 'README.md'))):
            return search_dir
        parent_dir = os.path.dirname(search_dir)
        if parent_dir == search_dir:
            break
        search_dir = parent_dir
    return os.getcwd()

project_root = find_project_root()
os.chdir(project_root)

DATA_PATH = 'data/processed/combined_stock_data.csv'
OUTPUT_DIR = 'data/features'

# Feature engineering parameters
LAG_PERIODS = [1, 2, 3, 5, 10, 20]
MA_WINDOWS = [5, 10, 20, 50]
VOL_WINDOWS = [5, 10, 20]

## Load Preprocessed Data

In [ ]:
# Load the preprocessed data
print("Loading preprocessed data...")
df = pd.read_csv(DATA_PATH)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['symbol', 'date']).reset_index(drop=True)

print(f"  Shape: {df.shape}")
print(f"  Stocks: {df['symbol'].unique()}")
print(f"  Date range: {df['date'].min()} to {df['date'].max()}")
print(f"\nFirst few rows:")
df.head(10)

## Task 1: Create Lagged Features

Lagged features capture historical patterns and momentum in stock prices.

In [ ]:
def create_lagged_features(df, lag_periods):
    """
    Create lagged return features for each stock.
    
    Args:
        df: DataFrame with stock data
        lag_periods: List of lag periods (e.g., [1, 5, 10])
    
    Returns:
        DataFrame with lagged features added
    """
    result_dfs = []
    
    for symbol in df['symbol'].unique():
        stock_df = df[df['symbol'] == symbol].copy()
        stock_df = stock_df.sort_values('date')
        
        # Create lagged returns
        for lag in lag_periods:
            stock_df[f'return_lag_{lag}'] = stock_df['log_return'].shift(lag)
        
        # Create lagged prices
        for lag in [1, 5, 10]:
            stock_df[f'close_lag_{lag}'] = stock_df['close'].shift(lag)
        
        # Create price ratios (current vs lagged)
        for lag in [5, 10, 20]:
            stock_df[f'price_ratio_{lag}'] = stock_df['close'] / stock_df['close'].shift(lag)
        
        result_dfs.append(stock_df)
    
    return pd.concat(result_dfs, axis=0).reset_index(drop=True)


print("Task 1: Creating lagged features...\n")
df_with_lags = create_lagged_features(df, LAG_PERIODS)

# Show new features
lag_features = [col for col in df_with_lags.columns if 'lag' in col or 'ratio' in col]
print(f"Created {len(lag_features)} lagged features:")
for feat in lag_features:
    print(f"  - {feat}")

print(f"\nDataFrame shape: {df_with_lags.shape}")
df_with_lags.head()

## Task 2: Technical Indicators

Technical indicators are widely used in trading to identify trends and momentum.

In [ ]:
def calculate_rsi(prices, period=14):
    """
    Calculate Relative Strength Index (RSI).
    RSI measures momentum and identifies overbought/oversold conditions.
    
    Args:
        prices: Series of closing prices
        period: Lookback period (default 14)
    
    Returns:
        RSI values (0-100)
    """
    delta = prices.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi


def calculate_macd(prices, fast=12, slow=26, signal=9):
    """
    Calculate MACD (Moving Average Convergence Divergence).
    MACD shows the relationship between two moving averages.
    
    Args:
        prices: Series of closing prices
        fast: Fast EMA period
        slow: Slow EMA period
        signal: Signal line period
    
    Returns:
        Tuple of (MACD line, Signal line, MACD histogram)
    """
    ema_fast = prices.ewm(span=fast, adjust=False).mean()
    ema_slow = prices.ewm(span=slow, adjust=False).mean()
    
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    macd_histogram = macd_line - signal_line
    
    return macd_line, signal_line, macd_histogram


def calculate_bollinger_bands(prices, window=20, num_std=2):
    """
    Calculate Bollinger Bands.
    Bollinger Bands measure volatility and identify overbought/oversold levels.
    
    Args:
        prices: Series of closing prices
        window: Moving average window
        num_std: Number of standard deviations
    
    Returns:
        Tuple of (upper band, middle band, lower band, bandwidth, %B)
    """
    middle_band = prices.rolling(window=window).mean()
    std = prices.rolling(window=window).std()
    
    upper_band = middle_band + (num_std * std)
    lower_band = middle_band - (num_std * std)
    
    # Bandwidth: measures volatility
    bandwidth = (upper_band - lower_band) / middle_band
    
    # %B: position within bands
    percent_b = (prices - lower_band) / (upper_band - lower_band)
    
    return upper_band, middle_band, lower_band, bandwidth, percent_b


def add_technical_indicators(df, ma_windows):
    """
    Add all technical indicators to the dataframe.
    
    Args:
        df: DataFrame with stock data
        ma_windows: List of moving average windows
    
    Returns:
        DataFrame with technical indicators added
    """
    result_dfs = []
    
    for symbol in df['symbol'].unique():
        stock_df = df[df['symbol'] == symbol].copy()
        stock_df = stock_df.sort_values('date')
        
        # Simple Moving Averages (SMA)
        for window in ma_windows:
            stock_df[f'sma_{window}'] = stock_df['close'].rolling(window=window).mean()
            # Price relative to SMA
            stock_df[f'price_to_sma_{window}'] = stock_df['close'] / stock_df[f'sma_{window}']
        
        # Exponential Moving Averages (EMA)
        for window in [12, 26]:
            stock_df[f'ema_{window}'] = stock_df['close'].ewm(span=window, adjust=False).mean()
        
        # RSI
        stock_df['rsi_14'] = calculate_rsi(stock_df['close'], period=14)
        
        # MACD
        macd_line, signal_line, macd_hist = calculate_macd(stock_df['close'])
        stock_df['macd'] = macd_line
        stock_df['macd_signal'] = signal_line
        stock_df['macd_histogram'] = macd_hist
        
        # Bollinger Bands
        bb_upper, bb_middle, bb_lower, bb_bandwidth, bb_percent = calculate_bollinger_bands(stock_df['close'])
        stock_df['bb_upper'] = bb_upper
        stock_df['bb_middle'] = bb_middle
        stock_df['bb_lower'] = bb_lower
        stock_df['bb_bandwidth'] = bb_bandwidth
        stock_df['bb_percent'] = bb_percent
        
        # Momentum indicators
        stock_df['momentum_5'] = stock_df['close'] - stock_df['close'].shift(5)
        stock_df['momentum_10'] = stock_df['close'] - stock_df['close'].shift(10)
        
        # Rate of Change (ROC)
        stock_df['roc_5'] = (stock_df['close'] - stock_df['close'].shift(5)) / stock_df['close'].shift(5)
        stock_df['roc_10'] = (stock_df['close'] - stock_df['close'].shift(10)) / stock_df['close'].shift(10)
        
        result_dfs.append(stock_df)
    
    return pd.concat(result_dfs, axis=0).reset_index(drop=True)


print("Task 2: Adding technical indicators...\n")
df_with_tech = add_technical_indicators(df_with_lags, MA_WINDOWS)

# Show new features
tech_features = [col for col in df_with_tech.columns if any(x in col for x in ['sma', 'ema', 'rsi', 'macd', 'bb_', 'momentum', 'roc'])]
print(f"Created {len(tech_features)} technical indicator features:")
for feat in tech_features:
    print(f"  - {feat}")

print(f"\nDataFrame shape: {df_with_tech.shape}")

## Task 3: Volatility and Volume-Based Metrics

Volatility measures risk and uncertainty, while volume indicates trading activity.

In [ ]:
def add_volatility_volume_features(df, vol_windows):
    """
    Add volatility and volume-based features.
    
    Args:
        df: DataFrame with stock data
        vol_windows: List of volatility calculation windows
    
    Returns:
        DataFrame with volatility and volume features added
    """
    result_dfs = []
    
    for symbol in df['symbol'].unique():
        stock_df = df[df['symbol'] == symbol].copy()
        stock_df = stock_df.sort_values('date')
        
        # Historical volatility (standard deviation of returns)
        for window in vol_windows:
            stock_df[f'volatility_{window}'] = stock_df['log_return'].rolling(window=window).std()
            # Annualized volatility (assuming 252 trading days)
            stock_df[f'volatility_{window}_annualized'] = stock_df[f'volatility_{window}'] * np.sqrt(252)
        
        # Parkinson's volatility (uses high-low range)
        for window in vol_windows:
            hl_ratio = np.log(stock_df['high'] / stock_df['low'])
            stock_df[f'parkinson_vol_{window}'] = np.sqrt(
                (hl_ratio ** 2).rolling(window=window).mean() / (4 * np.log(2))
            )
        
        # Garman-Klass volatility (uses OHLC)
        for window in [10, 20]:
            hl = np.log(stock_df['high'] / stock_df['low']) ** 2
            co = np.log(stock_df['close'] / stock_df['open']) ** 2
            stock_df[f'gk_vol_{window}'] = np.sqrt(
                (0.5 * hl - (2 * np.log(2) - 1) * co).rolling(window=window).mean()
            )
        
        # Average True Range (ATR) - measure of volatility
        high_low = stock_df['high'] - stock_df['low']
        high_close = np.abs(stock_df['high'] - stock_df['close'].shift())
        low_close = np.abs(stock_df['low'] - stock_df['close'].shift())
        true_range = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
        stock_df['atr_14'] = true_range.rolling(window=14).mean()
        
        # Volume features
        for window in [5, 10, 20]:
            # Average volume
            stock_df[f'avg_volume_{window}'] = stock_df['volume'].rolling(window=window).mean()
            # Volume ratio (current vs average)
            stock_df[f'volume_ratio_{window}'] = stock_df['volume'] / stock_df[f'avg_volume_{window}']
        
        # Volume-weighted features
        stock_df['vwap_5'] = (
            (stock_df['close'] * stock_df['volume']).rolling(window=5).sum() / 
            stock_df['volume'].rolling(window=5).sum()
        )
        stock_df['price_to_vwap'] = stock_df['close'] / stock_df['vwap_5']
        
        # On-Balance Volume (OBV) - cumulative volume indicator
        obv = [0]
        for i in range(1, len(stock_df)):
            if stock_df['close'].iloc[i] > stock_df['close'].iloc[i-1]:
                obv.append(obv[-1] + stock_df['volume'].iloc[i])
            elif stock_df['close'].iloc[i] < stock_df['close'].iloc[i-1]:
                obv.append(obv[-1] - stock_df['volume'].iloc[i])
            else:
                obv.append(obv[-1])
        stock_df['obv'] = obv
        stock_df['obv_ema'] = stock_df['obv'].ewm(span=10, adjust=False).mean()
        
        # Intraday range features
        stock_df['daily_range'] = (stock_df['high'] - stock_df['low']) / stock_df['low']
        stock_df['close_to_high'] = (stock_df['high'] - stock_df['close']) / stock_df['high']
        stock_df['close_to_low'] = (stock_df['close'] - stock_df['low']) / stock_df['low']
        
        result_dfs.append(stock_df)
    
    return pd.concat(result_dfs, axis=0).reset_index(drop=True)


print("Task 3: Adding volatility and volume features...\n")
df_with_vol = add_volatility_volume_features(df_with_tech, VOL_WINDOWS)

# Show new features
vol_features = [col for col in df_with_vol.columns if any(x in col for x in ['volatility', 'vol_', 'atr', 'volume', 'vwap', 'obv', 'range', 'close_to'])]
print(f"Created {len(vol_features)} volatility and volume features:")
for feat in vol_features:
    print(f"  - {feat}")

print(f"\nDataFrame shape: {df_with_vol.shape}")

## Task 4: Feature Standardization and Normalization

Standardize features to have mean=0 and std=1 for better model performance.

## Feature Summary and Quality Check

In [ ]:
print("=" * 70)
print("Feature Engineering Summary")
print("=" * 70)

# Count features by category
all_features = df_with_vol.columns.tolist()
base_cols = ['date', 'open', 'high', 'low', 'close', 'volume', 'symbol', 'log_return', 'next_day_return']
engineered_features = [col for col in all_features if col not in base_cols]

print(f"\nTotal columns: {len(all_features)}")
print(f"Base columns: {len(base_cols)}")
print(f"Engineered features: {len(engineered_features)}")

# Categorize features
feature_categories = {
    'Lagged Returns': [f for f in engineered_features if 'return_lag' in f],
    'Lagged Prices': [f for f in engineered_features if 'close_lag' in f],
    'Price Ratios': [f for f in engineered_features if 'price_ratio' in f or 'price_to' in f],
    'Moving Averages': [f for f in engineered_features if 'sma' in f or 'ema' in f],
    'Momentum Indicators': [f for f in engineered_features if 'rsi' in f or 'momentum' in f or 'roc' in f],
    'MACD': [f for f in engineered_features if 'macd' in f],
    'Bollinger Bands': [f for f in engineered_features if 'bb_' in f],
    'Volatility': [f for f in engineered_features if 'volatility' in f or 'vol_' in f or 'atr' in f],
    'Volume': [f for f in engineered_features if 'volume' in f or 'vwap' in f or 'obv' in f],
    'Intraday': [f for f in engineered_features if 'range' in f or 'close_to' in f],
}

print("\nFeatures by category:")
for category, features in feature_categories.items():
    print(f"\n{category}: {len(features)} features")
    for feat in features[:5]:  # Show first 5
        print(f"  - {feat}")
    if len(features) > 5:
        print(f"  ... and {len(features) - 5} more")

# Check for missing values
print("\n" + "=" * 70)
print("Missing Values Analysis")
print("=" * 70)

missing_counts = df_with_vol[engineered_features].isnull().sum()
missing_pct = (missing_counts / len(df_with_vol) * 100)
missing_summary = pd.DataFrame({
    'Missing Count': missing_counts,
    'Missing %': missing_pct
})
missing_summary = missing_summary[missing_summary['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

if len(missing_summary) > 0:
    print(f"\nFeatures with missing values: {len(missing_summary)}")
    print(missing_summary.head(10))
else:
    print("\nNo missing values found!")

# Sample data preview
print("\n" + "=" * 70)
print("Sample Data (AAPL)")
print("=" * 70)
aapl_sample = df_with_vol[df_with_vol['symbol'] == 'AAPL'].tail(5)
print(aapl_sample[['date', 'close', 'next_day_return', 'rsi_14', 'macd', 'volatility_10', 'volume_ratio_5']].to_string())

In [ ]:
## Task 4: Feature Standardization and Normalization

Standardize features to have mean=0 and std=1 for better model performance.

def scale_features_by_stock(df):
    """Scale features separately for each stock using RobustScaler."""
    exclude_cols = ['date', 'symbol', 'next_day_return']
    features_to_scale = [col for col in df.columns if col not in exclude_cols]
    
    result_dfs = []
    scalers = {}
    
    for symbol in df['symbol'].unique():
        stock_df = df[df['symbol'] == symbol].copy()
        stock_df = stock_df.sort_values('date')
        
        scaler = RobustScaler()
        scaler.fit(stock_df[features_to_scale])
        
        scaled_features = pd.DataFrame(
            scaler.transform(stock_df[features_to_scale]),
            index=stock_df.index,
            columns=[f"{col}_scaled" for col in features_to_scale]
        )
        
        stock_df = pd.concat([stock_df, scaled_features], axis=1)
        scalers[symbol] = scaler
        result_dfs.append(stock_df)
    
    return pd.concat(result_dfs, axis=0).reset_index(drop=True), scalers

# Scale features
df_scaled, scalers = scale_features_by_stock(df_clean)

In [ ]:
## Task 5: Create Final Feature Matrix (X) and Target Vector (y)

Prepare the final dataset for model training.

def create_feature_matrix(df):
    """Create final feature matrix (X) and target vector (y) for each stock."""
    feature_cols = [col for col in df.columns if col.endswith('_scaled')]
    target_col = 'next_day_return'
    
    stock_data = {}
    for symbol in df['symbol'].unique():
        stock_df = df[df['symbol'] == symbol].copy()
        stock_df = stock_df.sort_values('date')
        
        valid_mask = stock_df[target_col].notnull()
        stock_df_clean = stock_df[valid_mask].copy()
        
        stock_data[symbol] = {
            'X': stock_df_clean[feature_cols].values,
            'y': stock_df_clean[target_col].values,
            'dates': stock_df_clean['date'].values,
            'feature_names': feature_cols,
            'n_samples': len(stock_df_clean),
            'n_features': len(feature_cols)
        }
    
    return stock_data

# Create feature matrices
stock_datasets = create_feature_matrix(df_scaled)

In [ ]:
## Save Feature-Engineered Data

import pickle

# Create output directory and save all files
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save complete dataset
df_scaled.to_csv(os.path.join(OUTPUT_DIR, "complete_features.csv"), index=False)

# Save individual stock datasets
for symbol in STOCK_LIST:
    stock_df = df_scaled[df_scaled['symbol'] == symbol]
    stock_df.to_csv(os.path.join(OUTPUT_DIR, f"{symbol}_features.csv"), index=False)

# Save feature matrices as numpy arrays
for symbol, data in stock_datasets.items():
    np.save(os.path.join(OUTPUT_DIR, f"{symbol}_X.npy"), data['X'])
    np.save(os.path.join(OUTPUT_DIR, f"{symbol}_y.npy"), data['y'])
    np.save(os.path.join(OUTPUT_DIR, f"{symbol}_dates.npy"), data['dates'])

# Save scalers and metadata
with open(os.path.join(OUTPUT_DIR, "scalers.pkl"), 'wb') as f:
    pickle.dump(scalers, f)

with open(os.path.join(OUTPUT_DIR, "feature_names.txt"), 'w') as f:
    for feat in stock_datasets['AAPL']['feature_names']:
        f.write(f"{feat}\n")

metadata = {
    'stocks': STOCK_LIST,
    'n_features': stock_datasets['AAPL']['n_features'],
    'feature_names': stock_datasets['AAPL']['feature_names'],
    'lag_periods': LAG_PERIODS,
    'ma_windows': MA_WINDOWS,
    'vol_windows': VOL_WINDOWS,
    'scaling_method': 'RobustScaler',
    'missing_value_strategy': 'drop_initial'
}

with open(os.path.join(OUTPUT_DIR, "metadata.pkl"), 'wb') as f:
    pickle.dump(metadata, f)

print("Feature engineering complete. Files saved to:", OUTPUT_DIR)